# Gemma 4

Prompting Gemma 4 with zero-shot or few-shot.

It supports images, text, video and audio.

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "google/gemma-4-E2B-it"
# Alternatives: google/gemma-4-E4B-it or google/gemma-4-26B-A4B-it

# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)

In [ ]:
SYSTEM_PROMPT = "You are an Artificial Intelligence for sexism detection and classification in social media contents."
SUBTASK_21 = "Is this meme sexist towards women? Answer just YES if you judge the meme as sexist towards women. Answer just NO if you judge the meme as not sexist towards women."
SUBTASK_22 = "Answer just DIRECT if the meme promotes or encourages sexism. Answer just JUDGEMENTAL if the meme criticises or condemns sexist behaviour."
SUBTASK_23 = '''Classify the given meme into one or more of these categories (multi-label allowed):
• IDEOLOGICAL-INEQUALITY if it rejects feminism or denies gender inequality.
• STEREOTYPING-DOMINANCE if it promotes traditional gender roles or male superiority.
• OBJECTIFICATION if it reduces women to appearance or sexualises them.
• SEXUAL-VIOLENCE if it contains sexual harassment or assault references.
• MISOGYNY-NON-SEXUAL-VIOLENCE if it expresses hatred or non-sexual violence
toward women.
The answer is just and strictly a list of strings, as the following example:
["CATEGORY_1", "CATEGORY_4"]'''
IMG = "https://nlp.uned.es/exist2026/media/sexist-ideological.jpg"
MAX_NEW_TOKENS = 2048

In [ ]:
messages = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": SYSTEM_PROMPT}
        ],
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "url": IMG},
            {"type": "text", "text": SUBTASK_21},
        ],
    },
]

In [ ]:
# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse output
processor.parse_response(response)